In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.ensemble import StackingRegressor


In [3]:
train_path = "train.csv"
test_path  = "test.csv"

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

train.head()


Train shape: (15000, 10)
Test shape : (10000, 9)


,id,Sex,Length,Diameter,Height,Weight,Shucked Weight,Viscera Weight,Shell Weight,Age
0,0,F,1.6000,1.2500,0.5750,43.700754,15.025235,10.262519,13.324265,17.0
1,1,M,1.5750,1.1625,0.3750,32.417653,12.927372,6.208540,8.930093,20.0
2,2,F,1.6125,1.2750,0.4375,34.373769,14.784264,9.142714,10.772810,10.0
3,3,M,1.2250,0.9875,0.3250,18.455524,8.037083,3.926406,4.819415,9.0
4,4,I,1.3750,1.0250,0.3875,22.736299,9.879801,5.513978,6.945627,10.0


In [4]:
train_cols = set(train.columns)
test_cols  = set(test.columns)

diff = list(train_cols - test_cols)

if len(diff) == 1:
    target_col = diff[0]
else:
    target_col = train.columns[-1]

print("Detected target column:", target_col)

# Optional safety check for this task
if target_col != "age":
    print("Warning: Detected target is not 'age'. Overriding to 'age' for submission format.")
    target_col = "age"

id_col = "id"
if id_col not in test.columns:
    raise ValueError(f"ID column '{id_col}' not found in test.csv")

print("Final target column:", target_col)
print("ID column:", id_col)


Detected target column: Age
Final target column: age
ID column: id


In [5]:
train_cols = set(train.columns)
test_cols  = set(test.columns)

diff = list(train_cols - test_cols)

if len(diff) == 1:
    target_col = diff[0]
else:
    extra = list(train_cols - test_cols)
    target_col = extra[0] if len(extra) > 0 else train.columns[-1]

print("Detected target column:", target_col)

id_col = "id"
if id_col not in test.columns:
    raise ValueError(f"ID column '{id_col}' not found in test.csv")

print("ID column:", id_col)


Detected target column: Age
ID column: id


In [ ]:
# Build feature and target sets
X = train.drop(columns=[target_col, id_col])
y = train[target_col]
test_features = test.drop(columns=[id_col])

In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "Valid:", X_valid.shape)


Train: (12000, 8) Valid: (3000, 8)


In [8]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)


Numeric features: 7
Categorical features: 1


In [9]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)


Numeric features: 7
Categorical features: 1


In [ ]:
base_models = [
    ("rf", RandomForestRegressor(
        n_estimators=1200, random_state=42, n_jobs=-1
    )),
    ("gbr", GradientBoostingRegressor(random_state=42)),
    ("hgb", HistGradientBoostingRegressor(random_state=42)),
    ("svr", SVR(C=10, epsilon=0.1))
]

final_model = Ridge(alpha=1.0, random_state=42)

stack = StackingRegressor(
    estimators=base_models,
    final_estimator=final_model,
    passthrough=False,
    n_jobs=-1
)


In [17]:
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("stack", stack)
])

model


,steps,"[('preprocess', ...), ('stack', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
valid_pred = model.predict(X_valid)
mae = mean_absolute_error(y_valid, valid_pred)

print(f"Validation MAE:model.fit(X_train, y_train) {mae:.6f}")


Validation MAE: 1.365539


In [19]:
cv_scores = cross_val_score(
    model,
    X, y,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

print(f"CV MAE (mean): {(-cv_scores).mean():.6f}")
print(f"CV MAE (std) : {(-cv_scores).std():.6f}")


CV MAE (mean): 1.349678
CV MAE (std) : 0.015381


In [20]:
model.fit(X, y)
test_pred = model.predict(test_features)

test_pred[:10]


array([13.82716183, 12.76146127,  9.56582054, 10.98103655,  9.32400712,
       10.03176969, 10.36881306,  7.97147124, 11.09399892, 11.31777777])

In [22]:
submission = pd.DataFrame({
    id_col: test[id_col],
    "age": test_pred   # force output column name
})

out_path = "submission.csv"
submission.to_csv(out_path, index=False)

print("Saved submission to:", out_path)
submission.head()


Saved submission to: submission.csv


,id,age
0,15000,13.827162
1,15001,12.761461
2,15002,9.565821
3,15003,10.981037
4,15004,9.324007
